## Hybrid Retriving using dense and Sparse Retriever

In [ ]:
# %pip install langchain==0.1.16 langchain-community==0.0.32
# %pip uninstall langchain #langchain-community -y
# %pip install langchain==0.1.16
%pip install faiss-cpu

In [ ]:
from langchain_community.vectorstores import FAISS

from langchain_huggingface import HuggingFaceEmbeddings
# from langchain_community.retrievers import ContextualCompressionRetriever
from langchain.retrievers import EnsembleRetriever
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document

In [ ]:
# Sample Documents
docs = [
    Document(page_content="Langchain helps build LLM applications"),
    Document(page_content="Pinecone is a vector database for semantic search."),
    Document(page_content="The Eiffel Tower is located in Paris."),
    Document(page_content="Langchain can be used to develop agentic AI applications."),
    Document(page_content="Langchain has many types of retrievers."),
]

## Dense Retriever (Chroma + HuggingFaceEmbeddings)
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2",
    # model_name="sentence-transformers/paraphrase-multilingual-MinLM-L12-v2",
    multi_process=True,
    show_progress=True,
    cache_folder="./../model_cache/",
)
dense_vector_store = Chroma.from_documents(docs, embedding_model, persist_directory="./../chroma_db/")
dense_retriever = dense_vector_store.as_retriever()

In [ ]:
### Sparse Retriever (BM25)
sparse_retriever = BM25Retriever.from_documents(docs)
sparse_retriever.k=3 ## top k documents to retrieve

## Step 4: Combine with ensemble retriever
hybrid_retriever = EnsembleRetriever(
    retrievers=[dense_retriever, sparse_retriever],
    weights=[0.7, 0.3],  # Adjust weights as needed
)

In [ ]:
hybrid_retriever

In [ ]:
## Step 5: Query and get results 
query = "How can I build an agentic AI application using LLMs?"
results = hybrid_retriever.invoke(query)

## Step 6: Print results
for i, doc in enumerate(results):
    print(f"\nDocuments {i+1}: {doc.page_content}")

## Usage as a RAG pipeline

In [ ]:
from langchain.chat_models import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain

In [ ]:
## Step 5: Prompt Template 
prompt = PromptTemplate.from_template("""
Answer the question based on the context below.

Context:
{context}

Question: {input}
""")

# Step 6: LLM 
# llm = ChatOpenAI(model="gpt-3.5-turbo", temperature=0.7)
llm = init_chat_model(model="gpt-3.5-turbo", temperature=0.2)
llm

In [ ]:
## Create Stuff Documents Chain to stuff all the relevent context that is coming from the retriever into the prompt and then pass it to the LLM
document_chain = create_stuff_documents_chain(llm=llm, prompt=prompt)
 
## create full RAG Chain by combining the retriever and the document chain
rag_chain = create_retrieval_chain(retriever=hybrid_retriever, combine_documents_chain=document_chain)
rag_chainddd

In [ ]:
## Test the RAG Chain
query = {"input": "How can I build an agentic AI application using LLMs?"}
response = rag_chain.invoke(query)

## output
print("Answer:\n",response["answer"])

print("\n\nSources Documents:")
for idx, doc in enumerate(response["context"]):
    print(f"\nDocument {idx + 1}:\n{doc.page_content}")